# California data preparation

Data preparation using field data from the ameriflux database and landsat acquisitions

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

In [ ]:
import extra.plot as plt
import extra.prepare as pr

In [ ]:
if os.environ.get("SWEAT_TEST_DATA_PATH", None) is None:
    msg = "Variable SWEAT_TEST_DATA_PATH must be set"
    raise ValueError(msg)

## Prepare input data

In [ ]:
stic_data_path = os.path.join(os.environ.get("SWEAT_TEST_DATA_PATH"), "stic")
input_data_path = os.path.join(
    stic_data_path, "California_2017-2025_PointScale.xlsx"
)

### Get information on input data

Input content

In [ ]:
info_df = pr.get_california_info(input_data_path)

In [ ]:
info_df

### Read data

In [ ]:
input_data = pr.read_california_data(input_data_path)

In [ ]:
input_data

### Prepare data

* Convert temperatures to Celsius degrees.
* Compute local time in seconds taking into account the Landsat's overpass over the area around 18h45 GMT, i.e. UTC.
* Compute long wave net radiation with the following formula :
$$
L_{net} = \epsilon * \sigma_{SB} * 1.24 * (ea/(T_a+273.15))^{1/7}*(T_a+273.15)^4 - \epsilon*\sigma_{SB}*(LST+273.15)^4
$$
* Compute net radiation from **ssr** and **lnet**

In [ ]:
data = pr.prepare_california_data(input_data)

In [ ]:
data.head()

In [ ]:
data["nir/swir"] = data["nir"] / data["swir"]
data["ts-ta"] = data["ts"] - data["ta"]

### Plot data

In [ ]:
plt.plot_data_for_sites(data, variable="fc")

In [ ]:
plt.plot_scatter_for_sites(data, x="ta", y="ts")

### Use meteorological data and radiation data from EC tower

In [ ]:
_, am_data = pr.get_ameriflux_data(data, stic_data_path)

In [ ]:
_, fl_data = pr.get_fluxnet_data(data, stic_data_path)

In [ ]:
corrected_data = pr.correct_data(
    data, am_data, variables=["ta", "td", "rh", "rn", "srn"], dataset="am"
)
corrected_data = pr.merge_data(
    corrected_data,
    am_data,
    variables=["le", "le_closed", "h", "g", "rn-g"],
    dataset="am",
)
corrected_data.to_csv(
    os.path.join(stic_data_path, "data_landsat_ameriflux.csv"), index=False
)

In [ ]:
corrected_data = pr.correct_data(
    data, fl_data, variables=["ta", "td", "rh", "rn", "srn"], dataset="fl"
)
corrected_data = pr.merge_data(
    corrected_data,
    fl_data,
    variables=["le", "le_closed", "h", "g", "rn-g"],
    dataset="fl",
)
corrected_data.to_csv(
    os.path.join(stic_data_path, "data_landsat_fluxnet.csv"), index=False
)

## Visualize data

In [ ]:
plt.plot_scatter_for_sites(corrected_data, x="ta", y="ts", axis_range=(0, 60))